In [1]:
# !pip install httpx
# !pip install scrapy

In [3]:
import pandas as pd

In [16]:
import os
import sys
import re

import arcpy
import pandas as pd
import geopandas as gpd

import json
import asyncio
import httpx

# set workspace folder
workspace = r'Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks'
arcpy.env.workspace = workspace

# set sys path for custom folder module

# Append the directory to the Python path
sys.path.append(workspace)

from utils import check_files_exist

In [17]:
# folder to create and folder input path for BBOX - envelope
folder_name = 'input_shp'

# directly run from separate module
folder_path = check_files_exist.create_folder(workspace,folder_name)
print(folder_path)

Folder 'input_shp' already exists in workspace: Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks
Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\input_shp


In [18]:
# shapefile name for input, to get boundary aoi
# shp_name = 'rectangle_aoi.shp'
shp_name = 'indexDraft250k.shp'
geometry_type = 'POLYGON'

In [19]:
file_path = os.path.join(folder_path,shp_name)

if not arcpy.Exists(file_path):
    # Create a new shapefile using CreateFeatureclass_management
    arcpy.management.CreateFeatureclass(
        out_path=folder_path,
        out_name=shp_name,
        geometry_type="POLYGON",
        template=None,
        has_m="DISABLED",
        has_z="DISABLED",
        spatial_reference='GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]];-400 -400 1000000000;-100000 10000;-100000 10000;8.98315284119521E-09;0.001;0.001;IsHighPrecision',
        config_keyword="",
        spatial_grid_1=0,
        spatial_grid_2=0,
        spatial_grid_3=0,
        out_alias=""
    )
else:
    print('file rectangle, already there')

file rectangle, already there


In [20]:
# description, properties of layer
desc = arcpy.Describe(file_path)

In [21]:
# envelope bbox for arcgis rest api
envelope = desc.extent

In [22]:
# print(envelope)
xmin, ymin, xmax, ymax = envelope.XMin, envelope.YMin, envelope.XMax, envelope.YMax

In [23]:
check_files_exist.create_folder(workspace,'input_json')

Folder 'input_json' already exists in workspace: Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks


'Z:\\GIS_ArcGISPro\\jupyter_notebook\\webgis_klhk\\arcgis_restapi_notebooks\\input_json'

In [24]:
# make a new name json, testing only
# json_file_name = 'bbox.json'
json_file_name = 'bbox_indo.json'

# adding path and defined, use hardcoded instead
json_file_path = workspace + "\\" + 'input_json' + "\\" + json_file_name

In [25]:
data = {}
data['xmin'] = xmin
data['ymin'] = ymin
data['xmax'] = xmax
data['ymax'] = ymax


# dictionary to json file
with open(json_file_path, 'w') as json_file:
    json.dump(data, json_file)

# '{xmin:107.593163470013,ymin:-7.13806689350134,xmax:107.685427020972,ymax:-7.06496818910933}' 

In [26]:
str(data)

"{'xmin': 90.00000000000006, 'ymin': -14.999999999999943, 'xmax': 144.0000000000001, 'ymax': 8.000000000000057}"

In [67]:
## AFTER EDITED in spider, to put in spatial envelope, let's run the command scrapy and check
# RUN THIS COMMAND IN ROOT, in the folder that has scrapy.cfg file
# scrapy crawl webgis_klhk_spider -O bbox_oid_list.json (or any other name .json)
# bbox_oid_list.json is  objectids that needed for request

In [2]:
### ARCPY SPATIAL ANALYSIS after DOWNLOADING FINISH and GET THE DATA NEEDED ####
aprx = arcpy.mp.ArcGISProject("CURRENT")
map = aprx.listMaps()[0]  # assumes data to be added to first map listed

workspace_scratch = r'Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\merged_MOEF.gdb'
arcpy.env.workspace = workspace_scratch

In [4]:
# get the list geojson from one folder, acquired after finished with scrapy task (not from this notebook)
def list_files(directory):
    files = []
    for file_name in os.listdir(directory):
        file_path = os.path.join(directory, file_name)
        if os.path.isfile(file_path):
            files.append(file_path)
    return files

list_data_files = list_files(r'Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json')

In [ ]:
# os.path.basename(list_data_files[0]).replace('.json','')

In [4]:
# for i in list_data_files:
#     print(i,'\n')

In [73]:
# for i in list_data_files:
#     filename = os.path.basename(i)
#     print(filename)
#     print(os.path.dirname(i), '\n')

In [5]:
# reconstruct list into nested list (grouping per layer)
list_data_files.sort() # this is important since the latter code is based on the sequential sorted naming (list)

# init the check name (hardcoded) can also coded with the name
# file_name_check = 'DEF_2003_2006'
# non hardcoded
filename = os.path.basename(list_data_files[0])

parts = filename.split('_')
if len(parts) >= 2:
    file_name_check = '_'.join(parts[:-1])
else:
    file_name_check = None

# file_name_check = filename[:len(filename)-7] # equal to 'DEF_2003_2006' in context the same value folder (above)

group_array = []
array = []

for i in range(len(list_data_files)):
    filename = os.path.basename(list_data_files[i])
    #filename_no_ext = filename[:len(test)-5]
    print(filename)
    
    #print(os.path.dirname(i), '\n')
    # check the name to add into array
    
    parts = filename.split('_')
    if len(parts) >= 2:
        file_name_check1 = '_'.join(parts[:-1])
    else:
        file_name_check1 = None
    
    if file_name_check == file_name_check1:
                
        # adding to array group if the same name before suffix, for merging later
        array += [list_data_files[i]]      
        
    else:
        # append into group_array, after added
        #print(group_array)
        # grouping the previous array (group) collection, nested list before reinit new group
        group_array.append(array)
        # re-init the starting array, to be added in group later
        array = [list_data_files[i]]
        # restart the file_name_check, regrouping later, assuming the list is already sorted
        if len(parts) >= 2:
            file_name_check = '_'.join(parts[:-1])
        else:
            file_name_check = None
        print(f'\n new group added {file_name_check}\n')
    
    # need to add array to group in the last index
    if i+1 == len(list_data_files):
        group_array.append(array)

DEF_2003_2006_1.json
DEF_2006_2009_1.json

 new group added DEF_2006_2009

DEF_2006_2009_2.json
DEF_2009_2011_1.json

 new group added DEF_2009_2011

DEF_2009_2011_2.json
DEF_2011_2012_1.json

 new group added DEF_2011_2012

DEF_2011_2012_2.json
DEF_2011_2012_3.json
DEF_2012_2013_1.json

 new group added DEF_2012_2013

DEF_2013_2014_1.json

 new group added DEF_2013_2014

DEF_2013_2014_2.json
DEF_2014_2015_1.json

 new group added DEF_2014_2015

DEF_2014_2015_2.json
DEF_2015_2016_1.json

 new group added DEF_2015_2016

DEF_2016_2017_1.json

 new group added DEF_2016_2017

DEF_2016_2017_2.json
DEF_2016_2017_3.json
DEF_2017_2018_1.json

 new group added DEF_2017_2018

DEF_2018_2019_1.json

 new group added DEF_2018_2019

DEF_2018_2019_2.json
DEF_2018_2019_3.json
DEF_2018_2019_4.json
DEF_2019_2020_1.json

 new group added DEF_2019_2020

DEF_2021_2022_1.json

 new group added DEF_2021_2022

PL_1990_1.json

 new group added PL_1990

PL_1990_2.json
PL_1990_3.json
PL_1990_4.json
PL_1990_5.jso

In [179]:
# for i in range(len(list_data_files)):
#     print(i)
#     print(len(list_data_files))

In [6]:
a = 0
for i in group_array:
    
    for j in i:
        print(j)
        a +=1
print(a)

Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\DEF_2003_2006_1.json
Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\DEF_2006_2009_1.json
Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\DEF_2006_2009_2.json
Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\DEF_2009_2011_1.json
Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\DEF_2009_2011_2.json
Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\DEF_2011_2012_1.json
Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\DEF_2011_2012_2.json
Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\DEF_2011_2012_3.json
Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\DEF_2012_2013_1.json
Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_not

In [11]:
# for i in group_array:
#     parts = os.path.basename(i[0]).split('_') # take only one in nested items, as an example for naming convention
#     if len(parts) >= 2:
#         file_name_check = '_'.join(parts[:-1])
#     else:
#         file_name_check = None
#     print(file_name_check)
#     print(os.path.join(r'Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\merged_MOEF.gdb',
#                                            file_name_check))

In [13]:
# iteration to create gdb, conversion json to gdb and merge them into one feature class
merged_layers = []
for i in group_array:
    print(f'processing {i}','\n')
    # initiate empty array, for grouping, and merging data-set later
    array_merge = []
    for j in i:
        arcpy.env.addOutputsToMap = False
        print(f' \n processing with the geojson files: {j}')
        # get the name base on the file name that must be unique
        output_feature_class = os.path.basename(j).replace('.json','')

        output_full_path = os.path.join(r'Z:\GIS_ArcGISPro\yt_DEMO\MyProject\MyProject.gdb', 
                                             output_feature_class)
        array_merge.append(output_full_path)

        print('conversion to gdb feature class')
        # iterate conversion json to features in arcpy
        arcpy.conversion.JSONToFeatures(
            in_json_file=j, # use the nested item take part of the group
            out_features=output_full_path,
            geometry_type="POLYGON" # hard coded into polygon, next time maybe need to change this if the feature is line, or point
        )
        print(f'converted to gdb feature class in {output_full_path}')
    
    parts = os.path.basename(i[0]).split('_') # take only one in nested items, as an example for naming convention
    if len(parts) >= 2:
        file_name_check = '_'.join(parts[:-1])
    else:
        file_name_check = None
    
    # need to be hard coded 
    output_full_path_merged = os.path.join(r'Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\merged_MOEF.gdb',
                                           file_name_check)
        
    # merge in the arrays, after added in nested for loop
    if len(array_merge) > 1:
        arcpy.env.addOutputsToMap = True
        print(f'merging features into one data to {output_full_path_merged} \n')
        arcpy.management.Merge(
            inputs=';'.join(array_merge),
            output=output_full_path_merged,
            add_source="NO_SOURCE_INFO"
        )
        
    # if there is only one in the data, then copy to the database, and rename accordingly
    else:
        #since after run, the data is not copy to the dataset
        arcpy.env.addOutputsToMap = True
        
        arcpy.management.Copy(
            in_data=array_merge[0],
            out_data=output_full_path_merged,
            data_type="FeatureClass",
            associated_data=None
        )
        print(f'only one feature, renaming instead and copy into {output_full_path_merged} \n')
        
    merged_layers.append(output_full_path_merged)

processing ['Z:\\GIS_ArcGISPro\\jupyter_notebook\\webgis_klhk\\arcgis_restapi_notebooks\\output_json\\DEF_2003_2006_1.json'] 

 
 processing with the geojson files: Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\DEF_2003_2006_1.json
conversion to gdb feature class
converted to gdb feature class in Z:\GIS_ArcGISPro\yt_DEMO\MyProject\MyProject.gdb\DEF_2003_2006_1
only one feature, renaming instead into Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\merged_MOEF.gdb\DEF_2003_2006 

processing ['Z:\\GIS_ArcGISPro\\jupyter_notebook\\webgis_klhk\\arcgis_restapi_notebooks\\output_json\\DEF_2006_2009_1.json', 'Z:\\GIS_ArcGISPro\\jupyter_notebook\\webgis_klhk\\arcgis_restapi_notebooks\\output_json\\DEF_2006_2009_2.json'] 

 
 processing with the geojson files: Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\DEF_2006_2009_1.json
conversion to gdb feature class
converted to gdb feature class in Z:\GIS_Ar

converted to gdb feature class in Z:\GIS_ArcGISPro\yt_DEMO\MyProject\MyProject.gdb\PL_1996_4
 
 processing with the geojson files: Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\PL_1996_5.json
conversion to gdb feature class
converted to gdb feature class in Z:\GIS_ArcGISPro\yt_DEMO\MyProject\MyProject.gdb\PL_1996_5
 
 processing with the geojson files: Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\PL_1996_6.json
conversion to gdb feature class
converted to gdb feature class in Z:\GIS_ArcGISPro\yt_DEMO\MyProject\MyProject.gdb\PL_1996_6
 
 processing with the geojson files: Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\PL_1996_7.json
conversion to gdb feature class
converted to gdb feature class in Z:\GIS_ArcGISPro\yt_DEMO\MyProject\MyProject.gdb\PL_1996_7
 
 processing with the geojson files: Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\PL_

 processing with the geojson files: Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\PL_2011_5.json
conversion to gdb feature class
converted to gdb feature class in Z:\GIS_ArcGISPro\yt_DEMO\MyProject\MyProject.gdb\PL_2011_5
 
 processing with the geojson files: Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\PL_2011_6.json
conversion to gdb feature class
converted to gdb feature class in Z:\GIS_ArcGISPro\yt_DEMO\MyProject\MyProject.gdb\PL_2011_6
 
 processing with the geojson files: Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\PL_2011_7.json
conversion to gdb feature class
converted to gdb feature class in Z:\GIS_ArcGISPro\yt_DEMO\MyProject\MyProject.gdb\PL_2011_7
 
 processing with the geojson files: Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\output_json\PL_2011_8.json
conversion to gdb feature class
converted to gdb feature class in Z:\GIS_ArcGISPro\

processing ['Z:\\GIS_ArcGISPro\\jupyter_notebook\\webgis_klhk\\arcgis_restapi_notebooks\\output_json\\PL_2020_1.json', 'Z:\\GIS_ArcGISPro\\jupyter_notebook\\webgis_klhk\\arcgis_restapi_notebooks\\output_json\\PL_2020_2.json', 'Z:\\GIS_ArcGISPro\\jupyter_notebook\\webgis_klhk\\arcgis_restapi_notebooks\\output_json\\PL_2020_3.json', 'Z:\\GIS_ArcGISPro\\jupyter_notebook\\webgis_klhk\\arcgis_restapi_notebooks\\output_json\\PL_2020_4.json', 'Z:\\GIS_ArcGISPro\\jupyter_notebook\\webgis_klhk\\arcgis_restapi_notebooks\\output_json\\PL_2020_5.json', 'Z:\\GIS_ArcGISPro\\jupyter_notebook\\webgis_klhk\\arcgis_restapi_notebooks\\output_json\\PL_2020_6.json', 'Z:\\GIS_ArcGISPro\\jupyter_notebook\\webgis_klhk\\arcgis_restapi_notebooks\\output_json\\PL_2020_7.json', 'Z:\\GIS_ArcGISPro\\jupyter_notebook\\webgis_klhk\\arcgis_restapi_notebooks\\output_json\\PL_2020_8.json', 'Z:\\GIS_ArcGISPro\\jupyter_notebook\\webgis_klhk\\arcgis_restapi_notebooks\\output_json\\PL_2020_9.json'] 

 
 processing with the 

In [16]:
merged_layers_clipped = []
for i in merged_layers:
    print(f'clipping the feature {os.path.basename(i)}')
    arcpy.analysis.Clip(
        in_features=i,
        clip_features=r"Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\input_shp\rectangle_aoi.shp",
        out_feature_class=i+ "_clip",
        cluster_tolerance=None
    )
    merged_layers_clipped.append(i+ "_clip")

clipping the feature DEF_2003_2006
clipping the feature DEF_2006_2009
clipping the feature DEF_2009_2011
clipping the feature DEF_2011_2012
clipping the feature DEF_2012_2013
clipping the feature DEF_2013_2014
clipping the feature DEF_2014_2015
clipping the feature DEF_2015_2016
clipping the feature DEF_2016_2017
clipping the feature DEF_2017_2018
clipping the feature DEF_2018_2019
clipping the feature DEF_2019_2020
clipping the feature DEF_2021_2022
clipping the feature PL_1990
clipping the feature PL_1996
clipping the feature PL_2000
clipping the feature PL_2003
clipping the feature PL_2006
clipping the feature PL_2009
clipping the feature PL_2011
clipping the feature PL_2012
clipping the feature PL_2013
clipping the feature PL_2014
clipping the feature PL_2015
clipping the feature PL_2016
clipping the feature PL_2020
clipping the feature PL_2021
clipping the feature PL_2022
clipping the feature REF_2017_2018
clipping the feature REF_2021_2022


In [37]:
# intersect, put the data into all one together for timeseries analysis later
# detect the requires data, PL_*_CLIP, listing the data in the gdb fc (feature classes)
gdb = r"Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\merged_MOEF.gdb"

# set the workspace first, to detect list of feature classes
arcpy.env.workspace = gdb

featureclasses = arcpy.ListFeatureClasses()

# Define a regular expression pattern to match the desired format
pattern = r"PL_\d{4}_clip"

# Use list comprehension to filter based on pattern PL_{yyyy}_clip
lc_moef = [item for item in featureclasses if re.match(pattern, item)]
lc_moef.sort()

In [39]:
# intersect, implementation
arcpy.analysis.Intersect(
    in_features=";".join(lc_moef),
    out_feature_class="PL_MOEF_1990_2022_aoi",
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="INPUT"
)

<Result 'Z:\\GIS_ArcGISPro\\jupyter_notebook\\webgis_klhk\\arcgis_restapi_notebooks\\merged_MOEF.gdb\\PL_MOEF_1990_2022_aoi'>

In [41]:
# dissolving, to clean the data
arcpy.management.Dissolve(
    in_features="PL_MOEF_1990_2022_aoi",
    out_feature_class="PL_MOEF_1990_2022_aoi_dissolved",
    dissolve_field="pl90_id;pl96_id;pl00_id;pl06_id;pl09_id;pl11_id;pl12_id;pl13_id;pl14_id;pl15_id;pl16_id;PL17_ID;PL_18_R;PL_19_R;pl2020_id;pl2021_id;pl2022_id",
    statistics_fields=None,
    multi_part="MULTI_PART",
    unsplit_lines="DISSOLVE_LINES",
    concatenation_separator=""
)

<Result 'Z:\\GIS_ArcGISPro\\jupyter_notebook\\webgis_klhk\\arcgis_restapi_notebooks\\merged_MOEF.gdb\\PL_MOEF_1990_2022_aoi_dissolved'>

In [42]:
# calculate area hectare in the new field
in_feature = 'PL_MOEF_1990_2022_aoi_dissolved'
output_field = "area_ha_geodesic"

# Use CalculateField to calculate the geodesic area in hectares
# Make sure you have a field that stores the shape geometry (SHAPE@)
expression = "!SHAPE!.getArea('GEODESIC', 'HECTARES')"
arcpy.CalculateField_management(in_feature, output_field, expression, "PYTHON3")

<Result 'PL_MOEF_1990_2022_aoi_dissolved'>

In [40]:
# # pre-downloaded, not from LC, just as backup
# gdb_location_lc = r'Z:\gdrive_treeo\GIS_data\LC MoEF 1990-2020\Landcover 1990-2020.gdb'

# feature_input = "LC_MoEF_1990_2020_Indonesia_fix"

# # map.addLayer(feature_input)

# arcpy.management.MakeFeatureLayer(gdb_location_lc+"\\"+feature_input, feature_input) 

# clip_shp_path = r'Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\input_shp\rectangle_aoi.shp'
# clip_shp_name = 'aoi_bbox_envelope'

# arcpy.management.MakeFeatureLayer(clip_shp_path,
#                                  clip_shp_name)


In [72]:
################## analysis pandas for tracking deforestation ##########################################

gdb = r"Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\merged_MOEF.gdb"

# set the workspace first, to detect list of feature classes
arcpy.env.workspace = gdb

# feature_class = in_feature
feature_class = 'PL_MOEF_1990_2022_aoi_dissolved' # hardcoded if you want to run directly  to this cell

# Create a list to store the field names
field_names = [field.name for field in arcpy.ListFields(feature_class)]

# Use arcpy.da.SearchCursor to retrieve the data
data = [row for row in arcpy.da.SearchCursor(feature_class, field_names)]

# Convert the data to a Pandas DataFrame
df = pd.DataFrame(data, columns=field_names)
 
df.head()

,OBJECTID,Shape,pl90_id,pl96_id,pl00_id,pl06_id,pl09_id,pl11_id,pl12_id,pl13_id,pl14_id,pl15_id,pl16_id,PL17_ID,PL_18_R,PL_19_R,pl2020_id,pl2021_id,pl2022_id,Shape_Length,Shape_Area,area_ha_geodesic
0,1,"(113.62511316959544, 0.046676048487836345)",2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001,2001,2001,2001.0,2001.0,2001.0,55.884323,1.403104,1727045.68027476
1,2,"(113.28742953823696, 0.20575113028899075)",2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001,2001,2001,2001.0,2001.0,2002.0,1.572863,0.004004,4928.37714646186
2,3,"(113.51589610015087, -0.07585273791688647)",2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001,2001,2001,2001.0,2001.0,2007.0,0.014600,0.000008,9.25863817467028
3,4,"(113.31199060946416, 0.3472646973776705)",2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001,2001,2001,2001.0,2001.0,2014.0,0.461352,0.000220,271.072395414539
4,5,"(113.10584608705565, 0.028599324756894622)",2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001.0,2001,2001,2001,2001.0,2001.0,5001.0,0.111336,0.000036,43.9326447201691


In [80]:
# reformating the dataframe
# avoid warning, unchained indexing, copy first
df_1 = df.copy()

# regex of pl* or PL*
check_columns = r'^(pl|PL)\d{2}'
pl_columns = df_1.filter(regex=check_columns).copy()

# set into int, first copy
pl_columns = pl_columns.astype(int)

# convert now
df_1.loc[:, pl_columns.columns] = pl_columns
df_1 = df_1.drop(columns=['Shape','Shape_Length','Shape_Area'])

# renaming column
columns_rename= {
    'PL17_ID': 'pl17_id',
    'PL_18_R': 'pl18_id',
    'PL_19_R':'pl19_id',
    'pl2020_id': 'pl20_id',
    'pl2021_id': 'pl21_id',
    'pl2022_id': 'pl22_id',   
}

df_1.rename(columns=columns_rename, inplace=True)

In [98]:
df_1.head()

,OBJECTID,pl90_id,pl96_id,pl00_id,pl06_id,pl09_id,pl11_id,pl12_id,pl13_id,pl14_id,pl15_id,pl16_id,pl17_id,pl18_id,pl19_id,pl20_id,pl21_id,pl22_id,area_ha_geodesic
0,1,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,1727045.68027476
1,2,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2002,4928.37714646186
2,3,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2007,9.25863817467028
3,4,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2014,271.072395414539
4,5,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,5001,43.9326447201691


In [76]:
# field_names
# data

In [83]:
# acquire the code of MoEF data (in arcgis pro), only absolute path works, unless specify arc.env.workspace
csv_code_moef = r'Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\Landcover Code MoEF.csv'

# read csv
df_code = pd.read_csv(csv_code_moef)

In [169]:
df_code

,kode,keterangan,Description,Description2
0,2001,Hutan lahan kering primer,Primary Dryland Forest,Forested land
1,2002,Hutan lahan kering sekunder / bekas tebangan,Secondary Dryland Forest,Forested land
2,2005,Hutan rawa primer,Primary Swamp Forest,Forested land
3,20051,Hutan rawa sekunder / bekas tebangan,Secondary Swamp Forest,Forested land
4,2004,Hutan mangrove primer,Primary Mangrove Forest,Forested land
5,20041,Hutan mangrove sekunder / bekas tebangan,Secondary Mangrove Forest,Forested land
6,2006,Hutan tanaman,Plantation forest,NaN
7,2010,Perkebunan / Kebun,Estate crop,NaN
8,2007,Semak belukar,Dry shrub,Degraded land
9,20071,Semak belukar rawa,Wet shrub,Degraded land


In [85]:
# we are going to mark the deforestation if change from forested land into other land
# list forested land is shown in df_code: 2001, 2002, 2005, 20051, 2004, 20041

# degraded forest on the other hand: from primary to secondary: 2001 -> 2002, 2002 -> 20051, 2004 -> 20041

In [111]:
# data cleaning first 
pl_columns = [col for col in df_1.columns if col.startswith('pl')]

# get unique code each columns
unique_values_dict = {}
for column in pl_columns:
    unique_values_dict[column] = df_1[column].unique()

# Print the unique values for each 'pl_' column
for column, values in unique_values_dict.items():
    print(f"Unique values for {column}:")
    print(values)

Unique values for pl90_id:
[ 2001  2002  2005  2006  2007  2010  2012  2014  5001 20051 20071 20091
 20092 20093 20122 20141 50011]
Unique values for pl96_id:
[ 2001  2002  2007  2014 20051 20071  2006  2010  2012 20091 20092 20141
 50011  5001 20093 20122]
Unique values for pl00_id:
[ 2001  2002  2014  2007 20051 20071  2006 20092  2010  2012 20091 20141
 50011  5001 20093 20122]
Unique values for pl06_id:
[ 2001  2002  2007  2014 20051 20071  2006  2010 20091 20092 20141  2012
 50011  5001 20093 20122]
Unique values for pl09_id:
[ 2001  2002  2007  2014 20092 20051 20071  2010 20091 20141  2006  2012
 50011  5001 20093 20122]
Unique values for pl11_id:
[ 2001  2002  2007  2014 20092 20051 20071  2010 20091 20141  2006  2012
 50011  5001 20093 20122]
Unique values for pl12_id:
[ 2001  2002  2007  2014 20092 20051 20071  2010 20141 20091  2006  2012
 50011  5001 20093 20122]
Unique values for pl13_id:
[ 2001  2007  2014  2002 20092 20071  2010 20091 20122 20141  2006  2012
 50011  5001

In [132]:
# error 0 code in pl15_id, pl17_id, checking the total area, if small areas, we can remove, if big area, that means topology correction should perform
df_filtered = df_1[(df_1['pl15_id'] == 0) | (df_1['pl17_id'] == 0)]

# converting type into float64
df_filtered = df_filtered.copy()
df_filtered['area_ha_geodesic'] = pd.to_numeric(
    df['area_ha_geodesic'], errors='coerce'
)
area_list = df_filtered['area_ha_geodesic'].to_list()

total_area = sum(area_list)
print(total_area) # the area is i believe its because the topological error, therefore we can remove)

0.0008637244840278445


In [134]:
list_to_remove = df_filtered['OBJECTID'].to_list()

In [135]:
list_to_remove

[208, 214, 232, 233, 234, 235, 236, 237, 238, 240, 2076, 2077, 2371, 2372, 2373, 2374, 2375, 2376, 2377, 2889, 2920, 3092, 3776]

In [138]:
len(list_to_remove)

23

In [136]:
df_1.shape

(4005, 19)

In [137]:
# removing oid from df_1 (data sources)
df_2 = df_1[~df_1['OBJECTID'].isin(list_to_remove)]
df_2.shape

(3982, 19)

In [140]:
df_2 = df_2.copy()
df_2['area_ha_geodesic'] = pd.to_numeric(
    df_2['area_ha_geodesic'], errors='coerce'
)

In [145]:
# check and recheck the existing code used in the dataframe (unique id)
# print(pl_columns)
# unique values each 'pl' column and store them in a list
unique_values_list =[]
for column in pl_columns:
    unique_values_list.extend(df_2[column].unique())
unique_code = unique_values_list
#unique_code

code_used = list(set(unique_code))
print(list(set(unique_code)))

[5001, 20121, 20122, 20141, 2001, 2002, 20051, 2005, 2006, 2007, 2010, 50011, 2012, 2014, 20071, 20091, 20092, 20093, 20094]


In [153]:
# melting the df, to restructuring the timeseries data alike, from column based to row
meltdf =  pd.melt(df_2, id_vars=['OBJECTID', 'area_ha_geodesic'], 
                  value_vars=['pl90_id', 'pl96_id', 'pl00_id', 
                              'pl06_id', 'pl09_id', 'pl11_id', 
                              'pl12_id', 'pl13_id', 'pl14_id', 
                              'pl15_id', 'pl16_id', 'pl17_id', 
                              'pl18_id', 'pl19_id', 'pl20_id', 
                              'pl21_id', 'pl22_id',])



In [155]:
meltdf.head()

,OBJECTID,area_ha_geodesic,variable,value
0,1,1.727046e+06,pl90_id,2001
1,2,4.928377e+03,pl90_id,2001
2,3,9.258638e+00,pl90_id,2001
3,4,2.710724e+02,pl90_id,2001
4,5,4.393264e+01,pl90_id,2001


In [226]:
# re- do the pivot table, change the structure back the code into column, but row index is time (pl_id)
Repiv = meltdf.pivot_table(values='area_ha_geodesic',index=['variable'],columns=['value'],aggfunc='sum', margins=True)
Repiv = Repiv.iloc[:-1]
Repiv = Repiv.rename(columns={"All":"Total_area_aoi"})
Repiv = Repiv.reset_index()

Repiv.fillna(0, inplace=True)

# Create a custom sorting order using Categorical
value_vars=['pl90_id', 'pl96_id', 'pl00_id', 
                              'pl06_id', 'pl09_id', 'pl11_id', 
                              'pl12_id', 'pl13_id', 'pl14_id', 
                              'pl15_id', 'pl16_id', 'pl17_id', 
                              'pl18_id', 'pl19_id', 'pl20_id', 
                              'pl21_id', 'pl22_id',]

Repiv['variable'] = pd.Categorical(Repiv['variable'], categories=value_vars, ordered=True)

# Sort the DataFrame based on the custom sorting order
Repiv = Repiv.sort_values(by='variable')

# Reset the index
Repiv = Repiv.reset_index(drop=True)

In [227]:
Repiv

value,variable,2001,2002,2005,2006,2007,2010,2012,2014,5001,20051,20071,20091,20092,20093,20094,20121,20122,20141,50011,Total_area_aoi
0,pl90_id,2.164561e+06,4.319349e+06,765.519695,11965.344190,9.316669e+05,13203.624265,11444.379331,128203.815802,31919.222853,138941.907602,2153.072174,33966.090781,4.803712e+05,1362.370301,0.000000,0.000000,3863.072510,16296.738933,2433.997732,8.292467e+06
1,pl96_id,2.124921e+06,4.232897e+06,0.000000,80740.503303,9.638632e+05,16828.208952,13545.442146,17218.395351,31898.058129,138645.428865,3386.632239,36640.148607,6.075876e+05,1362.370301,0.000000,0.000000,3863.072510,17036.692768,2033.315463,8.292467e+06
2,pl00_id,2.118305e+06,4.229050e+06,0.000000,82144.554383,9.716180e+05,16828.208952,13130.459826,18525.673717,31898.058129,138795.876176,3290.142450,37055.130927,6.075312e+05,1362.370301,0.000000,0.000000,3863.072510,17036.692768,2033.315463,8.292467e+06
3,pl06_id,1.873372e+06,4.414177e+06,0.000000,85527.738318,9.928511e+05,24301.761698,13545.442146,38595.226707,31898.058129,124395.606706,16105.575690,43349.150294,6.091919e+05,1362.370301,0.000000,0.000000,3863.072510,17897.921043,2033.315463,8.292467e+06
4,pl09_id,1.871662e+06,4.335469e+06,0.000000,85309.606498,1.028197e+06,57898.809239,13545.442146,39748.929664,31898.058129,100271.515533,34512.680127,45273.921332,6.224821e+05,1362.370301,0.000000,0.000000,3863.072510,18938.675120,2033.315463,8.292467e+06
5,pl11_id,1.863087e+06,4.257175e+06,0.000000,84593.098805,1.095658e+06,62251.589560,13139.917081,57970.447567,31898.058129,92823.329937,37003.219090,45313.838147,6.241882e+05,1362.370301,0.000000,0.000000,3863.072510,20106.607964,2033.315463,8.292467e+06
6,pl12_id,1.862137e+06,4.223707e+06,0.000000,84317.878199,1.113736e+06,62281.138273,13139.917081,70193.413901,31898.058129,91569.186150,36305.113252,45313.838147,6.283067e+05,1362.370301,0.000000,0.000000,3863.072510,22303.950007,2033.315463,8.292467e+06
7,pl13_id,1.861688e+06,4.194104e+06,0.000000,83737.781770,1.125170e+06,65711.398401,13139.917081,70282.539513,31898.058129,90160.090124,36976.825388,45508.280482,6.440466e+05,1362.370301,0.000000,0.000000,4133.371385,22514.898280,2033.315463,8.292467e+06
8,pl14_id,1.841792e+06,4.183171e+06,0.000000,84608.608581,1.142986e+06,71190.236796,13149.031553,70601.406328,31898.092742,90127.015906,36883.552938,45756.856216,6.483138e+05,1362.370301,0.000000,0.000000,4133.371385,24270.798049,2223.477417,8.292467e+06
9,pl15_id,1.828453e+06,4.171265e+06,0.000000,83908.208864,6.237895e+05,108000.284429,7332.943576,62744.344705,31941.577860,89032.215220,32552.113404,49544.034662,1.153401e+06,1350.493059,0.000000,0.000000,2699.229449,44250.346965,2202.893408,8.292467e+06


In [273]:
# Repiv[2001]
Repiv.columns

Index([      'variable',             2001,             2002,             2005,
                   2006,             2007,             2010,             2012,
                   2014,             5001,            20051,            20071,
                  20091,            20092,            20093,            20094,
                  20121,            20122,            20141,            50011,
       'Total_area_aoi'],
      dtype='object', name='value')

In [212]:
list_codes = [2001,             2002,             2005,
                   2006,             2007,             2010,             2012,
                   2014,             5001,            20051,            20071,
                  20091,            20092,            20093,            20094,
                  20121,            20122,            20141,            50011]

value_colors = {
    2001: '#6CDA11',
    2002: '#B3F830',
    2005: '#C5DE76',
    2006: '#FFA500',
    2007: '#FFD700',
    2010: '#FF5733',
    2012: '#6A5ACD',
    2014: '#8B4513',
    5001: '#4682B4',
    20051: '#32CD32',
    20071: '#D2691E',
    20091: '#FF1493',
    20092: '#FF4500',
    20093: '#87CEFA',
    20094: '#20B2AA',
    20121: '#4B0082',
    20122: '#800080',
    20141: '#FF69B4',
    50011: '#808080'
}

df_code_aoi = df_code[df_code['kode'].isin(list_codes)]

df_code_aoi = df_code_aoi.copy()

df_code_aoi['kode'] = pd.Categorical(df_code_aoi['kode'], categories=list_codes, ordered=True)

# Sort the DataFrame based on the custom sorting order
df_code_aoi = df_code_aoi.sort_values(by='kode')

# Reset the index
df_code_aoi = df_code_aoi.reset_index(drop=True)

list_code_desc = df_code_aoi['Description'].to_list()

dict_pair_code = {list_codes[i]:list_code_desc[i] for i in range(len(list_codes))}


In [209]:
dict_pair_code

{2001: 'Primary Dryland Forest ', 2002: 'Secondary Dryland Forest ', 2005: 'Primary Swamp Forest ', 2006: 'Plantation forest ', 2007: 'Dry shrub ', 2010: 'Estate crop ', 2012: 'Settlement areas ', 2014: 'Bare ground ', 5001: 'Open water ', 20051: 'Secondary Swamp Forest ', 20071: 'Wet shrub ', 20091: 'Pure dry agriculture ', 20092: 'Mixed dry agriculture ', 20093: 'Paddy Field ', 20094: 'Fish pond/aquaculture ', 20121: 'Port and harbor ', 20122: 'Transmigration areas ', 20141: 'Mining areas ', 50011: 'Open swamp '}

In [210]:
df_code_aoi

,kode,keterangan,Description,Description2
0,2001,Hutan lahan kering primer,Primary Dryland Forest,Forested land
1,2002,Hutan lahan kering sekunder / bekas tebangan,Secondary Dryland Forest,Forested land
2,2005,Hutan rawa primer,Primary Swamp Forest,Forested land
3,2006,Hutan tanaman,Plantation forest,NaN
4,2007,Semak belukar,Dry shrub,Degraded land
5,2010,Perkebunan / Kebun,Estate crop,NaN
6,2012,Permukiman / Lahan terbangun,Settlement areas,NaN
7,2014,Lahan terbuka,Bare ground,Degraded land
8,5001,Tubuh air,Open water,NaN
9,20051,Hutan rawa sekunder / bekas tebangan,Secondary Swamp Forest,Forested land


In [191]:
print(list_remark_lc)

['Bare ground ', 'Dry shrub ', 'Estate crop ', 'Fish pond/aquaculture ', 'Mining areas ', 'Mixed dry agriculture ', 'Open swamp ', 'Open water ', 'Paddy Field ', 'Plantation forest ', 'Port and harbor ', 'Primary Dryland Forest ', 'Primary Swamp Forest ', 'Pure dry agriculture ', 'Secondary Dryland Forest ', 'Secondary Swamp Forest ', 'Settlement areas ', 'Transmigration areas ', 'Wet shrub ']


In [211]:
len(list_codes)

19

In [177]:
df_code_aoi.shape

(19, 4)

In [183]:
df_code_aoi.columns

Index(['kode', 'keterangan', 'Description', 'Description2'], dtype='object')

In [231]:
import seaborn as sns
import matplotlib.pyplot as plt

x = Repiv['variable']

# Create a custom sorting order using Categorical
column_names =list_codes

y = [Repiv[field] for field in column_names]


plt.figure(figsize=(20,10))
pal = [v for k,v in value_colors.items()]
plt.stackplot(x,y ,labels=[v for k,v in dict_pair_code.items()],colors=pal) #alpha=0.4)
plt.legend(loc='upper right')
plt.xticks(x, rotation='vertical')
plt.title("Landcover MOEF 1990-2022")
plt.show()


In [225]:
Repiv

value,variable,2001,2002,2005,2006,2007,2010,2012,2014,5001,20051,20071,20091,20092,20093,20094,20121,20122,20141,50011,Total_area_aoi
0,pl90_id,2.164561e+06,4.319349e+06,765.519695,11965.344190,9.316669e+05,13203.624265,11444.379331,128203.815802,31919.222853,138941.907602,2153.072174,33966.090781,4.803712e+05,1362.370301,0.000000,0.000000,3863.072510,16296.738933,2433.997732,8.292467e+06
1,pl96_id,2.124921e+06,4.232897e+06,0.000000,80740.503303,9.638632e+05,16828.208952,13545.442146,17218.395351,31898.058129,138645.428865,3386.632239,36640.148607,6.075876e+05,1362.370301,0.000000,0.000000,3863.072510,17036.692768,2033.315463,8.292467e+06
2,pl00_id,2.118305e+06,4.229050e+06,0.000000,82144.554383,9.716180e+05,16828.208952,13130.459826,18525.673717,31898.058129,138795.876176,3290.142450,37055.130927,6.075312e+05,1362.370301,0.000000,0.000000,3863.072510,17036.692768,2033.315463,8.292467e+06
3,pl06_id,1.873372e+06,4.414177e+06,0.000000,85527.738318,9.928511e+05,24301.761698,13545.442146,38595.226707,31898.058129,124395.606706,16105.575690,43349.150294,6.091919e+05,1362.370301,0.000000,0.000000,3863.072510,17897.921043,2033.315463,8.292467e+06
4,pl09_id,1.871662e+06,4.335469e+06,0.000000,85309.606498,1.028197e+06,57898.809239,13545.442146,39748.929664,31898.058129,100271.515533,34512.680127,45273.921332,6.224821e+05,1362.370301,0.000000,0.000000,3863.072510,18938.675120,2033.315463,8.292467e+06
5,pl11_id,1.863087e+06,4.257175e+06,0.000000,84593.098805,1.095658e+06,62251.589560,13139.917081,57970.447567,31898.058129,92823.329937,37003.219090,45313.838147,6.241882e+05,1362.370301,0.000000,0.000000,3863.072510,20106.607964,2033.315463,8.292467e+06
6,pl12_id,1.862137e+06,4.223707e+06,0.000000,84317.878199,1.113736e+06,62281.138273,13139.917081,70193.413901,31898.058129,91569.186150,36305.113252,45313.838147,6.283067e+05,1362.370301,0.000000,0.000000,3863.072510,22303.950007,2033.315463,8.292467e+06
7,pl13_id,1.861688e+06,4.194104e+06,0.000000,83737.781770,1.125170e+06,65711.398401,13139.917081,70282.539513,31898.058129,90160.090124,36976.825388,45508.280482,6.440466e+05,1362.370301,0.000000,0.000000,4133.371385,22514.898280,2033.315463,8.292467e+06
8,pl14_id,1.841792e+06,4.183171e+06,0.000000,84608.608581,1.142986e+06,71190.236796,13149.031553,70601.406328,31898.092742,90127.015906,36883.552938,45756.856216,6.483138e+05,1362.370301,0.000000,0.000000,4133.371385,24270.798049,2223.477417,8.292467e+06
9,pl15_id,1.828453e+06,4.171265e+06,0.000000,83908.208864,6.237895e+05,108000.284429,7332.943576,62744.344705,31941.577860,89032.215220,32552.113404,49544.034662,1.153401e+06,1350.493059,0.000000,0.000000,2699.229449,44250.346965,2202.893408,8.292467e+06


In [233]:
# renaming column
columns_rename= {
    'pl90_id': 1990, 
    'pl96_id': 1996,
    'pl00_id': 2000, 
    'pl06_id': 2006, 
    'pl09_id': 2009, 
    'pl11_id': 2011, 
    'pl12_id': 2012, 
    'pl13_id': 2013, 
    'pl14_id': 2014, 
    'pl15_id': 2015, 
    'pl16_id': 2016, 
    'pl17_id': 2017, 
    'pl18_id': 2018, 
    'pl19_id': 2019, 
    'pl20_id': 2020, 
    'pl21_id': 2021, 
    'pl22_id': 2022,
}

df_2.rename(columns=columns_rename, inplace=True)

In [237]:
[v for k,v in columns_rename.items()]

[1990, 1996, 2000, 2006, 2009, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]

In [235]:
df_2.head()

,OBJECTID,1990,1996,2000,2006,2009,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,area_ha_geodesic
0,1,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,1.727046e+06
1,2,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2002,4.928377e+03
2,3,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2007,9.258638e+00
3,4,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2014,2.710724e+02
4,5,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,5001,4.393264e+01


In [277]:
# assuming there is no peat land in the area, 
# (need to ensure though in reality, intersecting/ overlaying those data peatland and land cover)
# if peatland vs mineral (dry) land data is found, we should group class, which land cover is forest vs no forest

forested_land = [2001,2002,2004,2005,20041,20051] # 2006 (plantation forest, is categorized as non-forest (avoid) deforestation definition)
non_forested_land = [2006,2007,2010,2012,2014,2500,3000,5001,20071,20091,20092,20093,20094,20121,20122,20141,50011]

def deforested(df,x0,x1):
    df[str(x1)+f"_Deforested"] = (df[x0].isin(forested_land)) & (df[x1].isin(non_forested_land))

X0 = [1990, 1996, 2000, 2006, 2009, 2011, 
      2012, 2013, 2014, 2015, 2016, 2017, 
      2018, 2019, 2020, 2021]
X1 = [1996, 2000, 2006, 2009, 2011, 
      2012, 2013, 2014, 2015, 2016, 2017, 
      2018, 2019, 2020, 2021, 2022]

for i in range(len(X0)):
#    print(X1[i])
#    df_calc[X1[i]+"_Deforested"] = deforested(df_calc,X0[i],X1[i])
    deforested(df_2,X0[i],X1[i])


In [278]:
df_2.head()

,OBJECTID,1990,1996,2000,2006,2009,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,area_ha_geodesic,1996_Deforested,2000_Deforested,2006_Deforested,2009_Deforested,2011_Deforested,2012_Deforested,2013_Deforested,2014_Deforested,2015_Deforested,2016_Deforested,2017_Deforested,2018_Deforested,2019_Deforested,2020_Deforested,2021_Deforested,2022_Deforested,type_land
0,1,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,1.727046e+06,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,dry_land
1,2,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2002,4.928377e+03,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,dry_land
2,3,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2007,9.258638e+00,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,dry_land
3,4,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2014,2.710724e+02,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,dry_land
4,5,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,5001,4.393264e+01,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,dry_land


In [279]:
df_2['type_land'] = 'dry_land' # need to adjust this by overlaying the area of peatland based on MoEF or Planology data

In [313]:
melt_def = pd.melt(df_2, id_vars=['OBJECTID','area_ha_geodesic'], value_vars= [str(x)+"_Deforested" for x in X1])
melt_def.head()

,OBJECTID,area_ha_geodesic,variable,value
0,1,1.727046e+06,1996_Deforested,False
1,2,4.928377e+03,1996_Deforested,False
2,3,9.258638e+00,1996_Deforested,False
3,4,2.710724e+02,1996_Deforested,False
4,5,4.393264e+01,1996_Deforested,False


In [315]:
melt_def.shape # breakdown the column into rows (melting)

(63712, 4)

In [281]:
Repiv_def = melt_def.pivot_table(values='area_ha_geodesic',index=['variable'],columns=['value'],aggfunc='sum', margins=True)
Repiv_def = Repiv_def.reset_index()

# indexing for sorting the variable name, deforestation from 1996 to 2022
indx = [str(x)+"_Deforested" for x in X1]# + ["All"]

# implementing the categorical, for sorting
Repiv_def['variable'] = pd.Categorical(Repiv_def['variable'],indx)
# apply the sorting based on categorical
Repiv_def = Repiv_def.sort_values(['variable'],ascending=[True]).reset_index(drop=True)

#Repiv_def = Repiv_def.iloc[:,:-1]

# renaming additional areas to confirm the data, total areas (total aoi)
Repiv_def = Repiv_def.rename(columns={"All":"Total Area"})

#Repiv_def.info()
#display(indx)
# NOTICE there is NAN in the data, that means, there is no change in the areas, 
# if we want to check remaining forest, refer to the Landcover data later,

Repiv_def = Repiv_def[Repiv_def['variable'].notna()]

display(Repiv_def)

value,variable,False,True,Total Area
0,1996_Deforested,8.164299e+06,128168.526130,8.292467e+06
1,2000_Deforested,8.281680e+06,10786.971727,8.292467e+06
2,2006_Deforested,8.218260e+06,74207.114144,8.292467e+06
3,2009_Deforested,8.187547e+06,104920.158768,8.292467e+06
4,2011_Deforested,8.197202e+06,95265.455733,8.292467e+06
5,2012_Deforested,8.254882e+06,37584.727831,8.292467e+06
6,2013_Deforested,8.260068e+06,32399.397943,8.292467e+06
7,2014_Deforested,8.261605e+06,30862.163260,8.292467e+06
8,2015_Deforested,8.266127e+06,26339.736856,8.292467e+06
9,2016_Deforested,8.214174e+06,78293.288796,8.292467e+06


In [282]:
def apply_year(df, column):
    x = int(column[:4])
    return x

Repiv_def['year'] = Repiv_def['variable'].apply(lambda x: apply_year(Repiv_def, x))
Repiv_def['area_deforested_ha'] = Repiv_def[True]

In [283]:
Repiv_def_fix = Repiv_def[['year','area_deforested_ha']]

In [284]:
display(Repiv_def_fix)

value,year,area_deforested_ha
0,1996,128168.526130
1,2000,10786.971727
2,2006,74207.114144
3,2009,104920.158768
4,2011,95265.455733
5,2012,37584.727831
6,2013,32399.397943
7,2014,30862.163260
8,2015,26339.736856
9,2016,78293.288796


In [285]:
indx

['1996_Deforested', '2000_Deforested', '2006_Deforested', '2009_Deforested', '2011_Deforested', '2012_Deforested', '2013_Deforested', '2014_Deforested', '2015_Deforested', '2016_Deforested', '2017_Deforested', '2018_Deforested', '2019_Deforested', '2020_Deforested', '2021_Deforested', '2022_Deforested']

In [286]:
# refer carbon - ghg - frel national 
# https://redd.unfccc.int/media/2nd_frl_indonesia_final_submit.pdf

frel_csv_data = r'Z:\GIS_ArcGISPro\jupyter_notebook\webgis_klhk\arcgis_restapi_notebooks\FREL_DATA - combined_database_for_csv.csv'

df_frel = pd.read_csv(frel_csv_data)
df_frel.head()

,land_cover_desc,land_cover_code,island_coverage,AGB_(Mg ha-1)_mean,AGB_(Mg ha-1)_SE,BGB_(Mg ha-1)_Mean,BGB_(Mg ha-1)_SE,Total_Ecosystem_(Mg ha-1)_Mean,Total_Ecosystem_(Mg ha-1)_SE,Uncertainty_percent
0,Primary \n Dryland \n Forest,2001,Bali Nusa Tenggara \n,280.45,11.69,81.33,3.39,361.78,12.17,6.6
1,Primary \n Dryland \n Forest,2001,Java,347.88,51.35,100.89,17.29,448.77,54.19,23.7
2,Primary \n Dryland \n Forest,2001,Kalimantan,325.90,10.05,94.51,2.89,420.41,10.45,4.9
3,Primary \n Dryland \n Forest,2001,Maluku,237.85,19.01,68.98,5.88,306.83,19.90,12.7
4,Primary \n Dryland \n Forest,2001,Papua,268.57,9.12,77.88,2.63,346.45,9.49,5.4


In [290]:
df_frel.columns

Index(['land_cover_desc', 'land_cover_code', 'island_coverage',
       'AGB_(Mg ha-1)_mean', 'AGB_(Mg ha-1)_SE', 'BGB_(Mg ha-1)_Mean',
       'BGB_(Mg ha-1)_SE', 'Total_Ecosystem_(Mg ha-1)_Mean',
       'Total_Ecosystem_(Mg ha-1)_SE', 'Uncertainty_percent'],
      dtype='object')

In [293]:
# examine the necessary data
forested_land = [2001,2002,2004,2005,20041,20051] # 2006 (plantation forest, is categorized as non-forest (avoid) deforestation definition)
non_forested_land = [2006,2007,2010,2012,2014,2500,3000,5001,20071,20091,20092,20093,20094,20121,20122,20141,50011]

# Create a filter based on the "code" column matching values in forested_land and "island_coverage" being 'Kalimantan'
# FILTER THIS based ISLAND KALIMANTAN, change it if neccessary
filter_condition = (df_frel['land_cover_code'].isin(forested_land)) & (df_frel['island_coverage'] == 'Kalimantan') | (df_frel['land_cover_code'].isin(non_forested_land))

# Apply the filter to the DataFrame
df_frel_filtered = df_frel[filter_condition].copy()

In [297]:
df_frel_filtered

,land_cover_desc,land_cover_code,island_coverage,AGB_(Mg ha-1)_mean,AGB_(Mg ha-1)_SE,BGB_(Mg ha-1)_Mean,BGB_(Mg ha-1)_SE,Total_Ecosystem_(Mg ha-1)_Mean,Total_Ecosystem_(Mg ha-1)_SE,Uncertainty_percent
2,Primary \n Dryland \n Forest,2001,Kalimantan,325.90,10.05,94.51,2.89,420.41,10.45,4.90
10,Secondary \n Dryland \n Forest,2002,Kalimantan,222.91,4.48,64.64,1.32,287.55,4.67,3.20
18,Primary \n Swamp \n Forest,2005,Kalimantan,285.09,24.16,62.72,7.10,347.81,25.18,14.20
26,Secondary \n Swamp \n Forest,20051,Kalimantan,215.71,7.38,47.46,1.83,263.17,7.60,5.70
42,Secondary \n Mangrove \n Forest,20041,Kalimantan,155.74,19.21,17.91,2.32,173.66,19.35,21.80
48,Plantation forest,2006,Indonesia (Average),75.78,7.52,24.63,2.44,100.40,7.91,15.44
49,Dry shrub,2007,Indonesia (Average),60.39,7.22,14.25,1.70,74.64,7.42,19.48
50,Estate crop,2010,Indonesia (Average),48.10,6.90,15.63,2.24,63.74,7.25,22.30
51,Settlement,2012,Indonesia (Average),2.17,1.17,0.63,0.34,2.80,1.21,85.18
52,Bare ground,2014,Indonesia (Average),2.40,1.36,0.57,0.32,2.97,1.39,92.17


In [307]:
df_frel_filtered_fix = df_frel_filtered[['land_cover_code','land_cover_desc','Total_Ecosystem_(Mg ha-1)_Mean']]

In [308]:
df_frel_filtered_tco2e = df_frel_filtered_fix.copy()

In [309]:
# converting biomass to carbon and to co2 equivalent
df_frel_filtered_tco2e['tCO2_per_ha'] = df_frel_filtered_tco2e['Total_Ecosystem_(Mg ha-1)_Mean'] * 3.67 * 0.47 

In [310]:
# this one only apply in Kalimantan region island (forest frel)
df_frel_filtered_tco2e

,land_cover_code,land_cover_desc,Total_Ecosystem_(Mg ha-1)_Mean,tCO2_per_ha
2,2001,Primary \n Dryland \n Forest,420.41,725.165209
10,2002,Secondary \n Dryland \n Forest,287.55,495.994995
18,2005,Primary \n Swamp \n Forest,347.81,599.937469
26,20051,Secondary \n Swamp \n Forest,263.17,453.941933
42,20041,Secondary \n Mangrove \n Forest,173.66,299.546134
48,2006,Plantation forest,100.40,173.179960
49,2007,Dry shrub,74.64,128.746536
50,2010,Estate crop,63.74,109.945126
51,2012,Settlement,2.80,4.829720
52,2014,Bare ground,2.97,5.122953


In [316]:
df_2.head()

,OBJECTID,1990,1996,2000,2006,2009,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,area_ha_geodesic,1996_Deforested,2000_Deforested,2006_Deforested,2009_Deforested,2011_Deforested,2012_Deforested,2013_Deforested,2014_Deforested,2015_Deforested,2016_Deforested,2017_Deforested,2018_Deforested,2019_Deforested,2020_Deforested,2021_Deforested,2022_Deforested,type_land
0,1,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,1.727046e+06,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,dry_land
1,2,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2002,4.928377e+03,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,dry_land
2,3,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2007,9.258638e+00,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,dry_land
3,4,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2014,2.710724e+02,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,dry_land
4,5,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,5001,4.393264e+01,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,dry_land


In [335]:
df_3 = df_2.copy()

def check_last_lc(df, x0, x_def1):
    last_lc = 0
    if df[x_def1]:
        last_lc = df[x0]
    return last_lc

X0 = [1990, 1996, 2000, 2006, 2009, 2011, 
      2012, 2013, 2014, 2015, 2016, 2017, 
      2018, 2019, 2020, 2021]
X1 = [1996, 2000, 2006, 2009, 2011, 
      2012, 2013, 2014, 2015, 2016, 2017, 
      2018, 2019, 2020, 2021, 2022]

x_def1 = [str(i)+'_Deforested' for i in X1]

df_3['last_lc'] = 0  # Initialize the 'last_lc' column with zeros

for i in range(len(X0)):
    df_3['last_lc'] = df_3.apply(lambda row: check_last_lc(row, X0[i], x_def1[i]) if row['last_lc'] == 0 else row['last_lc'], axis=1)

In [338]:
df_3.head()

,OBJECTID,1990,1996,2000,2006,2009,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,area_ha_geodesic,1996_Deforested,2000_Deforested,2006_Deforested,2009_Deforested,2011_Deforested,2012_Deforested,2013_Deforested,2014_Deforested,2015_Deforested,2016_Deforested,2017_Deforested,2018_Deforested,2019_Deforested,2020_Deforested,2021_Deforested,2022_Deforested,type_land,last_lc
0,1,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,1.727046e+06,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,dry_land,0
1,2,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2002,4.928377e+03,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,dry_land,0
2,3,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2007,9.258638e+00,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,dry_land,2001
3,4,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2014,2.710724e+02,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,dry_land,2001
4,5,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,2001,5001,4.393264e+01,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,dry_land,2001


In [340]:
# # check re check
# df_3[(df_3['1996_Deforested']== True) & (df_3['last_lc']==0)] # re-check on the bug, but handled already

In [327]:
# df_3[df_3[2021]==2002] # just to check it works, uncomment first

In [341]:
# re assign the total areas deforestation based on the forest type, and 
# deforestation -> frel value into 0 (emission activity)
# degradation (TODO) -> frel value of forest type, substract secondary type (primary to secondary)
# peatland decomposition (TODO)

melt_def = pd.melt(df_3, id_vars=['OBJECTID','last_lc','area_ha_geodesic'], value_vars= [str(x)+"_Deforested" for x in X1])
melt_def.head()

,OBJECTID,last_lc,area_ha_geodesic,variable,value
0,1,0,1.727046e+06,1996_Deforested,False
1,2,0,4.928377e+03,1996_Deforested,False
2,3,2001,9.258638e+00,1996_Deforested,False
3,4,2001,2.710724e+02,1996_Deforested,False
4,5,2001,4.393264e+01,1996_Deforested,False


In [346]:
Repiv_def_2 = melt_def.pivot_table(values='area_ha_geodesic',index=['variable','last_lc'],columns=['value'],aggfunc='sum', margins=True)
Repiv_def_2 = Repiv_def_2.reset_index()

# indexing for sorting the variable name, deforestation from 1996 to 2022
indx = [str(x)+"_Deforested" for x in X1]# + ["All"]

# implementing the categorical, for sorting
Repiv_def_2['variable'] = pd.Categorical(Repiv_def_2['variable'],indx)
# apply the sorting based on categorical
Repiv_def_2 = Repiv_def_2.sort_values(['variable'],ascending=[True]).reset_index(drop=True)

#Repiv_def = Repiv_def.iloc[:,:-1]

# renaming additional areas to confirm the data, total areas (total aoi)
Repiv_def_2 = Repiv_def_2.rename(columns={"All":"Total Area"})

#Repiv_def.info()
#display(indx)
# NOTICE there is NAN in the data, that means, there is no change in the areas, 
# if we want to check remaining forest, refer to the Landcover data later,

Repiv_def_2 = Repiv_def_2[Repiv_def_2['variable'].notna()]

# display(Repiv_def_2)

In [347]:
def apply_year(df, column):
    x = int(column[:4])
    return x

Repiv_def_2['year'] = Repiv_def_2['variable'].apply(lambda x: apply_year(Repiv_def_2, x))
Repiv_def_2['area_deforested_ha'] = Repiv_def_2[True]

In [351]:
Repiv_def_2

value,variable,last_lc,False,True,Total Area,year,area_deforested_ha
0,1996_Deforested,0,7.440680e+06,NaN,7.440680e+06,1996,NaN
1,1996_Deforested,2001,6.960040e+03,2269.454132,9.229494e+03,1996,2269.454132
2,1996_Deforested,2002,6.563872e+05,124757.621101,7.811448e+05,1996,124757.621101
3,1996_Deforested,2005,NaN,765.519695,7.655197e+02,1996,765.519695
4,1996_Deforested,20051,6.027194e+04,375.931202,6.064787e+04,1996,375.931202
...,...,...,...,...,...,...,...
75,2022_Deforested,2005,7.655197e+02,NaN,7.655197e+02,2022,NaN
76,2022_Deforested,0,7.440680e+06,NaN,7.440680e+06,2022,NaN
77,2022_Deforested,2001,8.804884e+03,424.610410,9.229494e+03,2022,424.610410
78,2022_Deforested,2002,7.641683e+05,16976.438070,7.811448e+05,2022,16976.438070


In [357]:
Repiv_def_2[(Repiv_def_2['last_lc']==20051) & (Repiv_def_2['area_deforested_ha'].isna())]

value,variable,last_lc,False,True,Total Area,year,area_deforested_ha
5,2000_Deforested,20051,60647.871347,NaN,60647.871347,2000,NaN


In [360]:
Repiv_def_2_fix = Repiv_def_2.copy()

Repiv_def_2_fix = Repiv_def_2[['variable','last_lc','area_deforested_ha']]

# temporary fix, re-occuring deforestation should treat differently
Repiv_def_2_fix = Repiv_def_2_fix[(Repiv_def_2_fix['last_lc']!=0) & (Repiv_def_2_fix['area_deforested_ha'].notna())]                     

In [361]:
Repiv_def_2_fix

value,variable,last_lc,area_deforested_ha
1,1996_Deforested,2001,2269.454132
2,1996_Deforested,2002,124757.621101
3,1996_Deforested,2005,765.519695
4,1996_Deforested,20051,375.931202
6,2000_Deforested,2002,10781.363585
9,2000_Deforested,2001,5.608142
11,2006_Deforested,2001,1442.009294
12,2006_Deforested,2002,58291.517112
14,2006_Deforested,20051,14473.587738
15,2009_Deforested,20051,24124.091173


In [362]:
reference_def_emission = pd.merge(Repiv_def_2_fix, df_frel_filtered_tco2e, left_on='last_lc', right_on='land_cover_code', 
                                  how='left')

In [366]:
reference_def_emission_total = reference_def_emission.copy()

reference_def_emission_total = reference_def_emission_total[['variable','last_lc', 
                                                             'tCO2_per_ha', 'area_deforested_ha']]
reference_def_emission_total['total_tco2e_emission'] = reference_def_emission_total['tCO2_per_ha'] * reference_def_emission_total['area_deforested_ha']

display(reference_def_emission_total)

,variable,last_lc,tCO2_per_ha,area_deforested_ha,total_tco2e_emission
0,1996_Deforested,2001,725.165209,2269.454132,1.645729e+06
1,1996_Deforested,2002,495.994995,124757.621101,6.187916e+07
2,1996_Deforested,2005,599.937469,765.519695,4.592639e+05
3,1996_Deforested,20051,453.941933,375.931202,1.706509e+05
4,2000_Deforested,2002,495.994995,10781.363585,5.347502e+06
5,2000_Deforested,2001,725.165209,5.608142,4.066829e+03
6,2006_Deforested,2001,725.165209,1442.009294,1.045695e+06
7,2006_Deforested,2002,495.994995,58291.517112,2.891230e+07
8,2006_Deforested,20051,453.941933,14473.587738,6.570168e+06
9,2009_Deforested,20051,453.941933,24124.091173,1.095094e+07


In [367]:
# reference_def_emission_total

In [222]:
# arcpy.analysis.Clip(
#     in_features=feature_input,
#     clip_features=clip_shp_name,
#     out_feature_class=workspace_scratch+"\\"+'AOI_LC_1990_2020_MoEF',
#     cluster_tolerance=None
# )

In [116]:
# !pip install geopandas

In [152]:
# import geopandas as gpd

# gdf = gpd.read_file(workspace+"\\"+'example_queries.json')
# gdf.to_file(workspace+"\\"+'example_queries.shp')

In [153]:
# aprx = arcpy.mp.ArcGISProject("CURRENT")
# map = aprx.listMaps()[0]  # assumes data to be added to first map listed

# location_shp = os.path.join(workspace,'example_queries.shp')

# map.addDataFromPath(location_shp)

In [51]:
# '''
# # for testing only, this will be used in scrapy, check in scrapy spider for the perusal

# # open the file of json and create dictionary object of it
# with open(workspace + "\\" + 'bbox_oid_list.json', 'r' ) as oid_file:
#     oid_dict = json.load(oid_file)

# # oid_dict
# print(len(oid_dict))

# # flatten, to avoid nested for loop
# flatten_dict_oid = {}
# for i in oid_dict:
#     for url, value in i.items():
#         flatten_dict_oid[url] = value

# # flatten_dict_oid
# print(len(flatten_dict_oid))
        

# for key, value in flatten_dict_oid.items():
#     list_oids = flatten_dict_oid[key]['objectIds']
#     if list_oids is not None:
#         oid_name = flatten_dict_oid[key]['objectIdFieldName']
#         chunk_size = 1000
#         query_chunks = []
#         for i in range(0, len(list_oids), chunk_size):
#             chunk = list_oids[i:i + chunk_size]
#             query = ' or '.join([f"{oid_name} = {str(oid)}" for oid in chunk])
#             query_chunks.append(query)
#         flatten_dict_oid[key]['query'] = query_chunks
        
# fix_dict = {}
# for key, value in flatten_dict_oid.items():
#     if flatten_dict_oid[key].get('query') is not None:
# #         a +=1
# #         print(a)
# #         print(key)
#         fix_dict[key] = value
    
# print(len(fix_dict))

# # fix_dict['/server/rest/services/Time_Series/PL_2014/MapServer/0']['query']

# # for i in fix_dict['/server/rest/services/Time_Series/PL_2014/MapServer/0']['query']:
# #     print(i)
# #     print('------- \n')

# example_query_1000id = fix_dict['/server/rest/services/Time_Series/PL_2014/MapServer/0']['query'][0]

# #acquired from browser, and transform to python dictionary
# params = {'where': 'objectid = 65605 or objectid = 83388 or objectid = 83516 or objectid = 90232 or objectid = 90553 or objectid = 91150 or objectid = 91355 or objectid = 91372 or objectid = 91373 or objectid = 91374 or objectid = 91380 or objectid = 91381 or objectid = 91384 or objectid = 91385 or objectid = 91386 or objectid = 91387 or objectid = 91388 or objectid = 91389 or objectid = 91390 or objectid = 91391 or objectid = 91392 or objectid = 91393 or objectid = 91394 or objectid = 91395 or objectid = 91396 or objectid = 91397 or objectid = 91398 or objectid = 91399 or objectid = 91401 or objectid = 91402 or objectid = 91403 or objectid = 91404 or objectid = 91405 or objectid = 91406 or objectid = 91407 or objectid = 91409 or objectid = 91410 or objectid = 91411 or objectid = 91412 or objectid = 91413 or objectid = 91414 or objectid = 91415 or objectid = 91416 or objectid = 91417 or objectid = 91418 or objectid = 91419 or objectid = 91420 or objectid = 91421 or objectid = 91422 or objectid = 91423 or objectid = 91424 or objectid = 91425 or objectid = 91426 or objectid = 91427 or objectid = 91428 or objectid = 91430 or objectid = 91431 or objectid = 91432 or objectid = 91433 or objectid = 91434 or objectid = 91435 or objectid = 91436 or objectid = 91437 or objectid = 91438 or objectid = 91439 or objectid = 91440 or objectid = 91441 or objectid = 91442 or objectid = 91443 or objectid = 91444 or objectid = 91445 or objectid = 91446 or objectid = 91447 or objectid = 91448 or objectid = 91449 or objectid = 91450 or objectid = 91451 or objectid = 91452 or objectid = 91453 or objectid = 91454 or objectid = 91455 or objectid = 91456 or objectid = 91457 or objectid = 91458 or objectid = 91459 or objectid = 91460 or objectid = 91461 or objectid = 91462 or objectid = 91463 or objectid = 91464 or objectid = 91465 or objectid = 91466 or objectid = 91467 or objectid = 91468 or objectid = 91469 or objectid = 91470 or objectid = 91471 or objectid = 91473 or objectid = 91474 or objectid = 91479 or objectid = 91480 or objectid = 91481 or objectid = 91482 or objectid = 91483 or objectid = 91484 or objectid = 91485 or objectid = 91486 or objectid = 91489 or objectid = 91490 or objectid = 91491 or objectid = 91492 or objectid = 91493 or objectid = 91494 or objectid = 91495 or objectid = 91496 or objectid = 91497 or objectid = 91498 or objectid = 91499 or objectid = 91500 or objectid = 91501 or objectid = 91509 or objectid = 91510 or objectid = 91511 or objectid = 91512 or objectid = 91513 or objectid = 91514 or objectid = 91515 or objectid = 91516 or objectid = 91517 or objectid = 91518 or objectid = 91519 or objectid = 91520 or objectid = 91521 or objectid = 91522 or objectid = 91523 or objectid = 91524 or objectid = 91525 or objectid = 91526 or objectid = 91527 or objectid = 91528 or objectid = 91529 or objectid = 91531 or objectid = 91532 or objectid = 91534 or objectid = 91535 or objectid = 91536 or objectid = 91538 or objectid = 91539 or objectid = 91540 or objectid = 91541 or objectid = 91542 or objectid = 91543 or objectid = 91546 or objectid = 91547 or objectid = 91548 or objectid = 91549 or objectid = 91550 or objectid = 91551 or objectid = 91553 or objectid = 91554 or objectid = 91555 or objectid = 91556 or objectid = 91557 or objectid = 91558 or objectid = 91559 or objectid = 91560 or objectid = 91561 or objectid = 91562 or objectid = 91563 or objectid = 91564 or objectid = 91565 or objectid = 91566 or objectid = 91567 or objectid = 91568 or objectid = 91569 or objectid = 91571 or objectid = 91572 or objectid = 91573 or objectid = 91574 or objectid = 91575 or objectid = 91576 or objectid = 91577 or objectid = 91578 or objectid = 91579 or objectid = 91580 or objectid = 91582 or objectid = 91583 or objectid = 91584 or objectid = 91585 or objectid = 91586 or objectid = 91587 or objectid = 91589 or objectid = 91597 or objectid = 91600 or objectid = 91601 or objectid = 91602 or objectid = 91603 or objectid = 91604 or objectid = 91605 or objectid = 91606 or objectid = 91607 or objectid = 91608 or objectid = 91609 or objectid = 91610 or objectid = 91611 or objectid = 91612 or objectid = 91613 or objectid = 91614 or objectid = 91615 or objectid = 91616 or objectid = 91617 or objectid = 91619 or objectid = 91620 or objectid = 91621 or objectid = 91622 or objectid = 91623 or objectid = 91624 or objectid = 91625 or objectid = 91626 or objectid = 91627 or objectid = 91628 or objectid = 91629 or objectid = 91631 or objectid = 91632 or objectid = 91633 or objectid = 91634 or objectid = 91635 or objectid = 91636 or objectid = 91637 or objectid = 91638 or objectid = 91639 or objectid = 91640 or objectid = 91642 or objectid = 91643 or objectid = 91644 or objectid = 91648 or objectid = 91650 or objectid = 91651 or objectid = 91652 or objectid = 91653 or objectid = 91654 or objectid = 91655 or objectid = 91656 or objectid = 91657 or objectid = 91658 or objectid = 91659 or objectid = 91660 or objectid = 91661 or objectid = 91666 or objectid = 91667 or objectid = 91668 or objectid = 91816 or objectid = 91941 or objectid = 91946 or objectid = 91947 or objectid = 91948 or objectid = 91950 or objectid = 91951 or objectid = 91952 or objectid = 91953 or objectid = 91954 or objectid = 91955 or objectid = 91956 or objectid = 91957 or objectid = 91958 or objectid = 91959 or objectid = 91960 or objectid = 91961 or objectid = 91962 or objectid = 91963 or objectid = 91965 or objectid = 91966 or objectid = 91967 or objectid = 91968 or objectid = 91969 or objectid = 91970 or objectid = 91971 or objectid = 91972 or objectid = 91973 or objectid = 91974 or objectid = 91975 or objectid = 91976 or objectid = 91977 or objectid = 91978 or objectid = 91979 or objectid = 91980 or objectid = 91981 or objectid = 91982 or objectid = 91983 or objectid = 91984 or objectid = 91985 or objectid = 91986 or objectid = 91988 or objectid = 91990 or objectid = 91991 or objectid = 91992 or objectid = 91993 or objectid = 91994 or objectid = 91995 or objectid = 91996 or objectid = 91997 or objectid = 91998 or objectid = 91999 or objectid = 92000 or objectid = 92001 or objectid = 92002 or objectid = 92003 or objectid = 92004 or objectid = 92006 or objectid = 92007 or objectid = 92008 or objectid = 92009 or objectid = 92010 or objectid = 92011 or objectid = 92012 or objectid = 92013 or objectid = 92014 or objectid = 92015 or objectid = 92016 or objectid = 92017 or objectid = 92018 or objectid = 92019 or objectid = 92021 or objectid = 92022 or objectid = 92023 or objectid = 92024 or objectid = 92026 or objectid = 92027 or objectid = 92028 or objectid = 92029 or objectid = 92030 or objectid = 92031 or objectid = 92032 or objectid = 92033 or objectid = 92034 or objectid = 92035 or objectid = 92036 or objectid = 92037 or objectid = 92038 or objectid = 92040 or objectid = 92041 or objectid = 92043 or objectid = 92044 or objectid = 92045 or objectid = 92046 or objectid = 92047 or objectid = 92050 or objectid = 92051 or objectid = 92052 or objectid = 92053 or objectid = 92054 or objectid = 92055 or objectid = 92056 or objectid = 92057 or objectid = 92058 or objectid = 92059 or objectid = 92060 or objectid = 92061 or objectid = 92062 or objectid = 92064 or objectid = 92065 or objectid = 92066 or objectid = 92067 or objectid = 92068 or objectid = 92069 or objectid = 92070 or objectid = 92071 or objectid = 92072 or objectid = 92073 or objectid = 92074 or objectid = 92075 or objectid = 92076 or objectid = 92077 or objectid = 92078 or objectid = 92079 or objectid = 92080 or objectid = 92081 or objectid = 92082 or objectid = 92083 or objectid = 92084 or objectid = 92085 or objectid = 92086 or objectid = 92087 or objectid = 92088 or objectid = 92090 or objectid = 92091 or objectid = 92093 or objectid = 92094 or objectid = 92095 or objectid = 92096 or objectid = 92097 or objectid = 92100 or objectid = 92101 or objectid = 92102 or objectid = 92104 or objectid = 92105 or objectid = 92106 or objectid = 92107 or objectid = 92108 or objectid = 92109 or objectid = 92110 or objectid = 92111 or objectid = 92112 or objectid = 92114 or objectid = 92115 or objectid = 92116 or objectid = 92117 or objectid = 92118 or objectid = 92119 or objectid = 92120 or objectid = 92121 or objectid = 92122 or objectid = 92123 or objectid = 92124 or objectid = 92125 or objectid = 92126 or objectid = 92127 or objectid = 92128 or objectid = 92129 or objectid = 92130 or objectid = 92131 or objectid = 92132 or objectid = 92133 or objectid = 92134 or objectid = 92135 or objectid = 92136 or objectid = 92137 or objectid = 92138 or objectid = 92139 or objectid = 92140 or objectid = 92141 or objectid = 92142 or objectid = 92143 or objectid = 92144 or objectid = 92145 or objectid = 92146 or objectid = 92147 or objectid = 92148 or objectid = 92149 or objectid = 92151 or objectid = 92152 or objectid = 92153 or objectid = 92154 or objectid = 92155 or objectid = 92156 or objectid = 92157 or objectid = 92158 or objectid = 92159 or objectid = 92160 or objectid = 92161 or objectid = 92162 or objectid = 92163 or objectid = 92164 or objectid = 92165 or objectid = 92166 or objectid = 92167 or objectid = 92168 or objectid = 92169 or objectid = 92170 or objectid = 92171 or objectid = 92172 or objectid = 92173 or objectid = 92174 or objectid = 92175 or objectid = 92176 or objectid = 92177 or objectid = 92178 or objectid = 92179 or objectid = 92180 or objectid = 92181 or objectid = 92182 or objectid = 92183 or objectid = 92184 or objectid = 92185 or objectid = 92186 or objectid = 92187 or objectid = 92188 or objectid = 92190 or objectid = 92191 or objectid = 92192 or objectid = 92193 or objectid = 92194 or objectid = 92195 or objectid = 92196 or objectid = 92197 or objectid = 92199 or objectid = 92200 or objectid = 92201 or objectid = 92203 or objectid = 92204 or objectid = 92205 or objectid = 92206 or objectid = 92207 or objectid = 92208 or objectid = 92209 or objectid = 92210 or objectid = 92211 or objectid = 92212 or objectid = 92213 or objectid = 92214 or objectid = 92215 or objectid = 92216 or objectid = 92217 or objectid = 92218 or objectid = 92220 or objectid = 92221 or objectid = 92222 or objectid = 92223 or objectid = 92224 or objectid = 92226 or objectid = 92227 or objectid = 92228 or objectid = 92229 or objectid = 92230 or objectid = 92231 or objectid = 92232 or objectid = 92233 or objectid = 92234 or objectid = 92235 or objectid = 92236 or objectid = 92237 or objectid = 92238 or objectid = 92239 or objectid = 92240 or objectid = 92241 or objectid = 92242 or objectid = 92243 or objectid = 92244 or objectid = 92246 or objectid = 92247 or objectid = 92248 or objectid = 92249 or objectid = 92250 or objectid = 92251 or objectid = 92252 or objectid = 92253 or objectid = 92255 or objectid = 92257 or objectid = 92258 or objectid = 92259 or objectid = 92260 or objectid = 92261 or objectid = 92262 or objectid = 92263 or objectid = 92264 or objectid = 92265 or objectid = 92266 or objectid = 92267 or objectid = 92268 or objectid = 92269 or objectid = 92270 or objectid = 92271 or objectid = 92272 or objectid = 92273 or objectid = 92274 or objectid = 92275 or objectid = 92276 or objectid = 92277 or objectid = 92278 or objectid = 92279 or objectid = 92280 or objectid = 92281 or objectid = 92282 or objectid = 92283 or objectid = 92285 or objectid = 92286 or objectid = 92287 or objectid = 92288 or objectid = 92289 or objectid = 92290 or objectid = 92291 or objectid = 92292 or objectid = 92293 or objectid = 92294 or objectid = 92295 or objectid = 92296 or objectid = 92297 or objectid = 92298 or objectid = 92299 or objectid = 92300 or objectid = 92302 or objectid = 92303 or objectid = 92304 or objectid = 92305 or objectid = 92306 or objectid = 92307 or objectid = 92308 or objectid = 92309 or objectid = 92310 or objectid = 92311 or objectid = 92312 or objectid = 92313 or objectid = 92315 or objectid = 92316 or objectid = 92317 or objectid = 92318 or objectid = 92319 or objectid = 92320 or objectid = 92321 or objectid = 92322 or objectid = 92323 or objectid = 92325 or objectid = 92326 or objectid = 92328 or objectid = 92329 or objectid = 92330 or objectid = 92331 or objectid = 92332 or objectid = 92333 or objectid = 92334 or objectid = 92335 or objectid = 92336 or objectid = 92337 or objectid = 92338 or objectid = 92339 or objectid = 92340 or objectid = 92341 or objectid = 92342 or objectid = 92343 or objectid = 92344 or objectid = 92345 or objectid = 92346 or objectid = 92347 or objectid = 92348 or objectid = 92349 or objectid = 92350 or objectid = 92351 or objectid = 92352 or objectid = 92353 or objectid = 92355 or objectid = 92357 or objectid = 92358 or objectid = 92359 or objectid = 92360 or objectid = 92361 or objectid = 92362 or objectid = 92363 or objectid = 92364 or objectid = 92365 or objectid = 92366 or objectid = 92367 or objectid = 92368 or objectid = 92369 or objectid = 92370 or objectid = 92371 or objectid = 92372 or objectid = 92373 or objectid = 92374 or objectid = 92376 or objectid = 92377 or objectid = 92378 or objectid = 92379 or objectid = 92380 or objectid = 92381 or objectid = 92382 or objectid = 92383 or objectid = 92384 or objectid = 92385 or objectid = 92386 or objectid = 92387 or objectid = 92388 or objectid = 92389 or objectid = 92390 or objectid = 92391 or objectid = 92392 or objectid = 92393 or objectid = 92394 or objectid = 92395 or objectid = 92396 or objectid = 92397 or objectid = 92398 or objectid = 92399 or objectid = 92400 or objectid = 92401 or objectid = 92402 or objectid = 92403 or objectid = 92404 or objectid = 92405 or objectid = 92406 or objectid = 92407 or objectid = 92408 or objectid = 92409 or objectid = 92410 or objectid = 92411 or objectid = 92412 or objectid = 92413 or objectid = 92414 or objectid = 92415 or objectid = 92416 or objectid = 92417 or objectid = 92419 or objectid = 92420 or objectid = 92421 or objectid = 92422 or objectid = 92423 or objectid = 92424 or objectid = 92425 or objectid = 92426 or objectid = 92427 or objectid = 92428 or objectid = 92429 or objectid = 92430 or objectid = 92431 or objectid = 92432 or objectid = 92433 or objectid = 92434 or objectid = 92435 or objectid = 92436 or objectid = 92437 or objectid = 92438 or objectid = 92439 or objectid = 92440 or objectid = 92441 or objectid = 92442 or objectid = 92445 or objectid = 92448 or objectid = 92449 or objectid = 92451 or objectid = 92452 or objectid = 92454 or objectid = 92455 or objectid = 92456 or objectid = 92457 or objectid = 92458 or objectid = 92459 or objectid = 92460 or objectid = 92461 or objectid = 92462 or objectid = 92463 or objectid = 92464 or objectid = 92465 or objectid = 92466 or objectid = 92467 or objectid = 92468 or objectid = 92469 or objectid = 92470 or objectid = 92471 or objectid = 92472 or objectid = 92473 or objectid = 92474 or objectid = 92475 or objectid = 92476 or objectid = 92477 or objectid = 92478 or objectid = 92479 or objectid = 92480 or objectid = 92481 or objectid = 92482 or objectid = 92483 or objectid = 92484 or objectid = 92485 or objectid = 92486 or objectid = 92487 or objectid = 92488 or objectid = 92489 or objectid = 92490 or objectid = 92491 or objectid = 92492 or objectid = 92493 or objectid = 92494 or objectid = 92495 or objectid = 92496 or objectid = 92498 or objectid = 92499 or objectid = 92500 or objectid = 92501 or objectid = 92502 or objectid = 92503 or objectid = 92504 or objectid = 92505 or objectid = 92506 or objectid = 92507 or objectid = 92508 or objectid = 92509 or objectid = 92510 or objectid = 92511 or objectid = 92512 or objectid = 92513 or objectid = 92514 or objectid = 92515 or objectid = 92516 or objectid = 92517 or objectid = 92518 or objectid = 92519 or objectid = 92520 or objectid = 92522 or objectid = 92523 or objectid = 92524 or objectid = 92525 or objectid = 92526 or objectid = 92527 or objectid = 92528 or objectid = 92529 or objectid = 92530 or objectid = 92531 or objectid = 92532 or objectid = 92533 or objectid = 92534 or objectid = 92535 or objectid = 92536 or objectid = 92537 or objectid = 92538 or objectid = 92539 or objectid = 92540 or objectid = 92541 or objectid = 92542 or objectid = 92543 or objectid = 92544 or objectid = 92545 or objectid = 92546 or objectid = 92547 or objectid = 92548 or objectid = 92549 or objectid = 92550 or objectid = 92551 or objectid = 92552 or objectid = 92565 or objectid = 92566 or objectid = 92567 or objectid = 92598 or objectid = 92599 or objectid = 92601 or objectid = 92602 or objectid = 92603 or objectid = 92604 or objectid = 92605 or objectid = 92606 or objectid = 92607 or objectid = 92608 or objectid = 92609 or objectid = 92610 or objectid = 92611 or objectid = 92612 or objectid = 92613 or objectid = 92614 or objectid = 92615 or objectid = 92616 or objectid = 92617 or objectid = 92618 or objectid = 92619 or objectid = 92620 or objectid = 92621 or objectid = 92622 or objectid = 92624 or objectid = 92625 or objectid = 92626 or objectid = 92627 or objectid = 92628 or objectid = 92629 or objectid = 92630 or objectid = 92631 or objectid = 92632 or objectid = 92633 or objectid = 92634 or objectid = 92635 or objectid = 92636 or objectid = 92637 or objectid = 92638 or objectid = 92639 or objectid = 92640 or objectid = 92641 or objectid = 92642 or objectid = 92643 or objectid = 92644 or objectid = 92645 or objectid = 92646 or objectid = 92647 or objectid = 92648 or objectid = 92649 or objectid = 92651 or objectid = 92652 or objectid = 92653 or objectid = 92654 or objectid = 92655 or objectid = 92656 or objectid = 92657 or objectid = 92658 or objectid = 92659 or objectid = 92660 or objectid = 92661 or objectid = 92662 or objectid = 92663 or objectid = 92664 or objectid = 92665 or objectid = 92666 or objectid = 92667 or objectid = 92668 or objectid = 92669 or objectid = 92670 or objectid = 92671 or objectid = 92672 or objectid = 92673 or objectid = 92674 or objectid = 92675 or objectid = 92676 or objectid = 92677 or objectid = 92678 or objectid = 92679 or objectid = 92680 or objectid = 92681 or objectid = 92682 or objectid = 92683 or objectid = 92684 or objectid = 92685 or objectid = 92686 or objectid = 92687 or objectid = 92688 or objectid = 92689 or objectid = 92690 or objectid = 92691 or objectid = 92692 or objectid = 92693 or objectid = 92694 or objectid = 92695 or objectid = 92696 or objectid = 92697 or objectid = 92698 or objectid = 92699 or objectid = 92700 or objectid = 92701 or objectid = 92702 or objectid = 92703 or objectid = 92704 or objectid = 92705 or objectid = 92706 or objectid = 92707 or objectid = 92708 or objectid = 92709 or objectid = 92710 or objectid = 92711 or objectid = 92712 or objectid = 92713 or objectid = 92714 or objectid = 92715 or objectid = 92716 or objectid = 92717 or objectid = 92718 or objectid = 92719 or objectid = 92720 or objectid = 92721 or objectid = 92722 or objectid = 92723 or objectid = 92724 or objectid = 92725 or objectid = 92726 or objectid = 92727 or objectid = 92728 or objectid = 92729 or objectid = 92730 or objectid = 92731 or objectid = 92732 or objectid = 92733 or objectid = 92734 or objectid = 92735 or objectid = 92736 or objectid = 92737 or objectid = 92738 or objectid = 92739 or objectid = 92740 or objectid = 92741 or objectid = 92742 or objectid = 92743 or objectid = 92744 or objectid = 92745 or objectid = 92746 or objectid = 92747 or objectid = 92748 or objectid = 92749 or objectid = 92750 or objectid = 92751 or objectid = 92752 or objectid = 92753 or objectid = 92754 or objectid = 92755 or objectid = 92756 or objectid = 92757 or objectid = 92758 or objectid = 92759 or objectid = 92760 or objectid = 92761 or objectid = 92762 or objectid = 92771 or objectid = 92772 or objectid = 92773 or objectid = 92774 or objectid = 92775 or objectid = 92776 or objectid = 92852 or objectid = 92895 or objectid = 92896 or objectid = 92897 or objectid = 92898 or objectid = 92899 or objectid = 92900 or objectid = 92901 or objectid = 92902 or objectid = 92903 or objectid = 92904 or objectid = 92905 or objectid = 92906',
# 'text': '',
# 'objectIds': '',
# 'time': '',
# 'timeRelation': 'esriTimeRelationOverlaps',
# 'geometry': '',
# 'geometryType': 'esriGeometryEnvelope',
# 'inSR': '',
# 'spatialRel': 'esriSpatialRelIntersects',
# 'distance': '',
# 'units': 'esriSRUnit_Foot',
# 'relationParam': '',
# 'outFields': '*',
# 'returnGeometry': 'true',
# 'returnTrueCurves': 'false',
# 'maxAllowableOffset': '',
# 'geometryPrecision': '',
# 'outSR': '',
# 'havingClause': '',
# 'returnIdsOnly': 'false',
# 'returnCountOnly': 'false',
# 'orderByFields': '',
# 'groupByFieldsForStatistics':  '',
# 'outStatistics': '',
# 'returnZ': 'false',
# 'returnM': 'false',
# 'gdbVersion': '',
# 'historicMoment': '',
# 'returnDistinctValues': 'false',
# 'resultOffset': '',
# 'resultRecordCount': '',
# 'returnExtentOnly': 'false',
# 'sqlFormat': 'none',
# 'datumTransformation': '',
# 'parameterValues': '',
# 'rangeValues': '',
# 'quantizationParameters': '',
# 'featureEncoding': 'esriDefault',
# 'f': 'geojson',
#     }
    
# '''

In [54]:
## hard way converting the list files of json
# # os.path.basename(list_data_files[0]).replace('.json','')
# # iteration to create gdb

# for i in list_data_files:
#     print(f' \n processing with the geojson files: {i}')
#     # get the name base on the file name that must be unique
#     output_feature_class = os.path.basename(i).replace('.json','')
    
#     # load the i (geojson file path)
#     with open(i, 'r') as geojson_file:
#         geojson_data = json.load(geojson_file)
        
#     # Extract the CRS information from the GeoJSON
#     crs_data = geojson_data.get('crs')  # Check if 'crs' key exists in GeoJSON
#     if crs_data:
#         crs_wkid = crs_data.get('properties', {}).get('name', '').split(':')[-1]
#     else:
#         # Default to WKID 4326 (WGS 1984) if CRS information is not found
#         crs_wkid = 4326
        
#     print(f'create a feature class of {i} into gdb')
#     # Create the feature class in the GDB with polygon geometry and specify the CRS
#     arcpy.management.CreateFeatureclass(arcpy.env.workspace, 
#                                         output_feature_class, 
#                                         'POLYGON', 
#                                         spatial_reference=arcpy.SpatialReference(int(crs_wkid)))
#     print('gdb created')
    
#     field_type_mapping = {
#         int: 'LONG',
#         float: 'DOUBLE',
#         str: 'TEXT',
#         bool: 'SHORT',
#     }
    
    
#     print('Check the structure of the data design')
    
#     # List of field names to skip when adding fields
#     skip_fields = ['objectid', 'shape_area', 'shape_length']
    
#     print('renaming the problematic names (invalid) character into _ from geojson, to make it work in arcgis gdb')
#     # Rename fields with problematic names before inserting
#     a = 0
#     for i, feature in enumerate(geojson_data['features']):
#         a += 1
#         properties = feature['properties']
#         for field in list(properties.keys()):
#             #print(f'check the field: {field}')
#             if '(' in field:
#                 print(f'replacing field {field}')
#                 # Replace problematic field names with an alternative name
#                 new_field_name = field.replace('(', '_').replace(')', '_').replace(' ', '_')
#                 print(f'new field name {new_field_name}')
#                 properties[new_field_name] = properties.pop(field)
#             elif field == 'objectid' or field == 'objectid_1':
#                 print(f'replacing field {field}')
#                 new_field_name = 'oid_moef'
#                 print(f'new field name {new_field_name}')
#                 properties[new_field_name] = properties.pop(field)
#         print(f'editing the geojson data feature {a}')
#         geojson_data['features'][i]['properties'] = properties

#     for key, value in geojson_data['features'][0]['properties'].items():
#         if key not in skip_fields:
#             # Determine the arcpy field type based on the Python data type
#             field_type = field_type_mapping[type(value)] if type(value) in field_type_mapping else 'TEXT'
#             arcpy.management.AddField(output_feature_class, key, field_type)

#     print('Insert data into fc with the cursor')

#     # Insert features into the feature class (matching field names)
#     with arcpy.da.InsertCursor(output_feature_class, list(geojson_data['features'][0]['properties'].keys())) as cursor:
#         a = 0
#         for feature in geojson_data['features']:
#             a += 1
#             coordinates = feature['geometry']['coordinates']
#             point_array = arcpy.Array([arcpy.Point(*coords) for coords in coordinates[0]])
#             polygon = arcpy.Polygon(point_array)
#             attributes = [polygon] + [feature['properties'][key] for key in geojson_data['features'][0]['properties'].keys()]
#             #print(f'inserting attribute: {attributes} into cursor')
#             print(f'inserting attribute number: {a} out of {len(geojson_data["features"])}')
#             cursor.insertRow(attributes)